# Baseline Forecasting

This notebook creates time-ordered data splits and evaluates simple persistence and historical-average baselines for hourly bike-share demand forecasting.

## 1. Connect Google Drive

Mount Drive and load the hourly demand matrix.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Load Hourly Demand Data

Load the 744-hour, 200-station demand matrix produced during preprocessing.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("/content/drive/MyDrive/bike_share_stgcn/data/processed")
HOURLY_DEMAND_FILE = PROCESSED_DIR / "hourly_pickup_demand_top_200.csv"

hourly_demand = pd.read_csv(
    HOURLY_DEMAND_FILE,
    index_col="timestamp",
    parse_dates=True
)

hourly_demand.columns = hourly_demand.columns.astype(str)

print(f"Demand matrix shape: {hourly_demand.shape}")
print(f"First timestamp: {hourly_demand.index.min()}")
print(f"Last timestamp: {hourly_demand.index.max()}")
print(f"Total pickups: {hourly_demand.to_numpy().sum():,}")

Demand matrix shape: (744, 200)
First timestamp: 2025-07-01 00:00:00
Last timestamp: 2025-07-31 23:00:00
Total pickups: 1,755,685


## 3. Create Time-Ordered Splits

Split the hourly data into 70% training, 10% validation, and 20% test periods without shuffling.

In [3]:
T = len(hourly_demand)

train_end = int(T * 0.70)
val_end = int(T * 0.80)

train_demand = hourly_demand.iloc[:train_end].copy()
val_demand = hourly_demand.iloc[train_end:val_end].copy()
test_demand = hourly_demand.iloc[val_end:].copy()

print(f"Training set:   {train_demand.shape} | {train_demand.index.min()} to {train_demand.index.max()}")
print(f"Validation set: {val_demand.shape} | {val_demand.index.min()} to {val_demand.index.max()}")
print(f"Test set:       {test_demand.shape} | {test_demand.index.min()} to {test_demand.index.max()}")

print(f"\nSplit sizes: {len(train_demand) / T:.1%} train, {len(val_demand) / T:.1%} validation, {len(test_demand) / T:.1%} test")

Training set:   (520, 200) | 2025-07-01 00:00:00 to 2025-07-22 15:00:00
Validation set: (75, 200) | 2025-07-22 16:00:00 to 2025-07-25 18:00:00
Test set:       (149, 200) | 2025-07-25 19:00:00 to 2025-07-31 23:00:00

Split sizes: 69.9% train, 10.1% validation, 20.0% test


## 4. Define Forecasting Settings

Use the previous 24 hours of demand to predict station demand 5, 10, and 15 hours ahead.

In [4]:
LOOKBACK_HOURS = 24
FORECAST_HORIZONS = [5, 10, 15]

print(f"Lookback window: {LOOKBACK_HOURS} hours")
print(f"Forecast horizons: {FORECAST_HORIZONS} hours")
print(f"Number of stations: {hourly_demand.shape[1]}")

Lookback window: 24 hours
Forecast horizons: [5, 10, 15] hours
Number of stations: 200


## 5. Define Evaluation Targets

Identify valid validation and test target times for each forecast horizon.

In [5]:
def valid_target_positions(split_start, split_end, lookback, horizon):
    """
    Return target positions whose forecasts use only past observations.
    """
    first_target = max(split_start, lookback + horizon)
    return np.arange(first_target, split_end)

split_positions = {
    "train": (0, train_end),
    "validation": (train_end, val_end),
    "test": (val_end, T),
}

target_positions = {}

for horizon in FORECAST_HORIZONS:
    target_positions[horizon] = {}

    for split_name, (split_start, split_end) in split_positions.items():
        positions = valid_target_positions(
            split_start=split_start,
            split_end=split_end,
            lookback=LOOKBACK_HOURS,
            horizon=horizon,
        )

        target_positions[horizon][split_name] = positions

        if len(positions) > 0:
            first_time = hourly_demand.index[positions[0]]
            last_time = hourly_demand.index[positions[-1]]

            print(
                f"{horizon:>2}-hour horizon | {split_name:<10}: "
                f"{len(positions):>3} targets | {first_time} to {last_time}"
            )

 5-hour horizon | train     : 491 targets | 2025-07-02 05:00:00 to 2025-07-22 15:00:00
 5-hour horizon | validation:  75 targets | 2025-07-22 16:00:00 to 2025-07-25 18:00:00
 5-hour horizon | test      : 149 targets | 2025-07-25 19:00:00 to 2025-07-31 23:00:00
10-hour horizon | train     : 486 targets | 2025-07-02 10:00:00 to 2025-07-22 15:00:00
10-hour horizon | validation:  75 targets | 2025-07-22 16:00:00 to 2025-07-25 18:00:00
10-hour horizon | test      : 149 targets | 2025-07-25 19:00:00 to 2025-07-31 23:00:00
15-hour horizon | train     : 481 targets | 2025-07-02 15:00:00 to 2025-07-22 15:00:00
15-hour horizon | validation:  75 targets | 2025-07-22 16:00:00 to 2025-07-25 18:00:00
15-hour horizon | test      : 149 targets | 2025-07-25 19:00:00 to 2025-07-31 23:00:00


## 6. Define Evaluation Metrics

Evaluate forecasts with MAE, RMSE, and masked MAPE, excluding zero-demand targets from MAPE.

In [6]:
def evaluate_forecasts(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    nonzero_mask = y_true > 0
    masked_mape = (
        np.mean(
            np.abs(
                (y_true[nonzero_mask] - y_pred[nonzero_mask])
                / y_true[nonzero_mask]
            )
        )
        * 100
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "Masked_MAPE_percent": masked_mape,
        "Nonzero_target_share_percent": nonzero_mask.mean() * 100,
    }

print("Evaluation function created.")

Evaluation function created.


## 7. Persistence Baseline

Use demand observed one forecast horizon earlier as the prediction.

In [7]:
demand_values = hourly_demand.to_numpy(dtype=np.float64)

persistence_results = []

for horizon in FORECAST_HORIZONS:
    test_positions = target_positions[horizon]["test"]

    y_true = demand_values[test_positions]
    y_pred = demand_values[test_positions - horizon]

    metrics = evaluate_forecasts(y_true, y_pred)

    persistence_results.append({
        "model": "Persistence",
        "horizon_hours": horizon,
        **metrics,
    })

persistence_results_df = pd.DataFrame(persistence_results)

display(persistence_results_df.round(3))

,model,horizon_hours,MAE,RMSE,Masked_MAPE_percent,Nonzero_target_share_percent
0,Persistence,5,11.286,15.245,205.850,90.493
1,Persistence,10,13.205,16.992,276.578,90.493
2,Persistence,15,12.864,16.629,238.756,90.493


### Result

The persistence baseline achieved test MAE values of 11.286, 13.205, and 12.864 for 5-, 10-, and 15-hour horizons, respectively. MAE and RMSE are treated as the primary metrics because percentage errors are sensitive to low demand counts.

## 8. Historical Hour-of-Week Baseline

Predict demand using each station's average demand at the same hour of the week in the training period.

In [8]:
train_with_time = train_demand.copy()
train_with_time["hour_of_week"] = (
    train_with_time.index.dayofweek * 24
    + train_with_time.index.hour
)

station_columns = hourly_demand.columns.tolist()

hour_of_week_means = (
    train_with_time
    .groupby("hour_of_week")[station_columns]
    .mean()
)

overall_station_means = train_demand.mean()

def historical_hour_of_week_predictions(target_times):
    target_hour_of_week = (
        target_times.dayofweek * 24
        + target_times.hour
    )

    predictions = []

    for how in target_hour_of_week:
        if how in hour_of_week_means.index:
            prediction = hour_of_week_means.loc[how].to_numpy()
        else:
            prediction = overall_station_means.to_numpy()

        predictions.append(prediction)

    return np.vstack(predictions)

historical_results = []

for horizon in FORECAST_HORIZONS:
    test_positions = target_positions[horizon]["test"]
    test_times = hourly_demand.index[test_positions]

    y_true = demand_values[test_positions]
    y_pred = historical_hour_of_week_predictions(test_times)

    metrics = evaluate_forecasts(y_true, y_pred)

    historical_results.append({
        "model": "Historical hour-of-week",
        "horizon_hours": horizon,
        **metrics,
    })

historical_results_df = pd.DataFrame(historical_results)

display(historical_results_df.round(3))

,model,horizon_hours,MAE,RMSE,Masked_MAPE_percent,Nonzero_target_share_percent
0,Historical hour-of-week,5,4.132,6.205,51.854,90.493
1,Historical hour-of-week,10,4.132,6.205,51.854,90.493
2,Historical hour-of-week,15,4.132,6.205,51.854,90.493


### Result

The historical hour-of-week baseline achieved a test MAE of 4.132 and RMSE of 6.205. Its scores are identical across horizons because predictions use only the target weekday and hour.

## 9. Save Baseline Results

Combine and save test results for the persistence and historical baselines.

In [9]:
OUTPUTS_DIR = Path("/content/drive/MyDrive/bike_share_stgcn/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = pd.concat(
    [persistence_results_df, historical_results_df],
    ignore_index=True
)

BASELINE_RESULTS_FILE = OUTPUTS_DIR / "baseline_test_results.csv"

baseline_results.to_csv(BASELINE_RESULTS_FILE, index=False)

display(
    baseline_results[
        ["model", "horizon_hours", "MAE", "RMSE", "Masked_MAPE_percent"]
    ].round(3)
)

print(f"Saved baseline results to: {BASELINE_RESULTS_FILE}")

,model,horizon_hours,MAE,RMSE,Masked_MAPE_percent
0,Persistence,5,11.286,15.245,205.850
1,Persistence,10,13.205,16.992,276.578
2,Persistence,15,12.864,16.629,238.756
3,Historical hour-of-week,5,4.132,6.205,51.854
4,Historical hour-of-week,10,4.132,6.205,51.854
5,Historical hour-of-week,15,4.132,6.205,51.854


Saved baseline results to: /content/drive/MyDrive/bike_share_stgcn/outputs/baseline_test_results.csv


## 10. Check Persistence Forecast Alignment

Verify that each persistence forecast uses demand observed exactly one forecast horizon earlier.

In [10]:
example_target_position = target_positions[10]["test"][0]

print(f"Target timestamp: {hourly_demand.index[example_target_position]}")
print(f"Target position: {example_target_position}\n")

for horizon in FORECAST_HORIZONS:
    source_position = example_target_position - horizon

    print(
        f"{horizon:>2}-hour horizon | "
        f"source timestamp: {hourly_demand.index[source_position]} | "
        f"time difference: "
        f"{hourly_demand.index[example_target_position] - hourly_demand.index[source_position]}"
    )

Target timestamp: 2025-07-25 19:00:00
Target position: 595

 5-hour horizon | source timestamp: 2025-07-25 14:00:00 | time difference: 0 days 05:00:00
10-hour horizon | source timestamp: 2025-07-25 09:00:00 | time difference: 0 days 10:00:00
15-hour horizon | source timestamp: 2025-07-25 04:00:00 | time difference: 0 days 15:00:00


### Check

Persistence predictions were verified to use observations exactly 5, 10, and 15 hours before each target. The non-monotonic persistence error across horizons therefore reflects demand patterns rather than an indexing error.